# 1 · Reproduce the headline result

**Runs on CPU in seconds. No GPU, no model download, no API key.**

The claim this notebook checks:

> Resampling importance is dominated neither by sentence **position** (the
> hypothesis I set out to test) nor by sentence **category** (the original
> paper's story), but by **how undecided the model still was** when the sentence
> arrived — the entropy of the answer distribution it lands in.

Everything below is computed from `data/sentences_*.csv`, which is committed.
If you only run one notebook, run this one.

In [1]:
import json, sys
from pathlib import Path
import numpy as np, pandas as pd
from scipy import stats

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "code"))
DATA = ROOT / "data"
TAG  = "DeepSeek-R1-Distill-Qwen-14B"

df      = pd.read_csv(DATA / f"sentences_{TAG}.csv")
summary = json.loads((DATA / f"summary_{TAG}.json").read_text())
print(f"{df.trace_id.nunique()} traces, {len(df)} sentences, "
      f"R = {summary['rollouts_per_prefix']} rollouts per prefix")

8 traces, 254 sentences, R = 64 rollouts per prefix


## 1.1 The variance decomposition

Position enters as a **cubic** so the comparison isn't rigged by giving category
a flexible functional form and position a straight line. Category enters as
dummies.

In [2]:
label_col = next(c for c in df.columns if c.startswith("label_"))
d = df.dropna(subset=["kl_resampling", "position", "entropy_before"]).copy()
y = d.kl_resampling.values.astype(float)

def r2(X):
    """R^2 of an OLS fit with intercept."""
    A = np.column_stack([np.ones(len(y))] + ([X] if X is not None else []))
    beta, *_ = np.linalg.lstsq(A, y, rcond=None)
    resid = y - A @ beta
    return 1 - (resid**2).sum() / ((y - y.mean())**2).sum()

pos  = d.position.values.astype(float)
P    = np.column_stack([pos, pos**2, pos**3])          # position, cubic
cats = sorted(d[label_col].unique())
D    = np.column_stack([(d[label_col] == c).astype(float) for c in cats[:-1]])
E    = d.entropy_before.values.astype(float).reshape(-1, 1)   # headroom

rows = [("position (cubic)",        r2(P)),
        ("sentence category",       r2(D)),
        ("headroom  H(A_{i-1})",    r2(E)),
        ("position + category",     r2(np.column_stack([P, D]))),
        ("headroom + position",     r2(np.column_stack([E, P]))),
        ("all three",               r2(np.column_stack([E, P, D])))]

print(f"n = {len(d)} sentences\n")
for name, v in rows:
    bar = "#" * int(v * 60)
    print(f"  {name:24s} R2 = {v:.3f}  {bar}")

uniq_E = r2(np.column_stack([E,P,D])) - r2(np.column_stack([P,D]))
uniq_D = r2(np.column_stack([E,P,D])) - r2(np.column_stack([E,P]))
print(f"\n  unique to headroom, over position+category : {uniq_E:+.3f}")
print(f"  unique to category, over headroom+position : {uniq_D:+.3f}")

n = 254 sentences

  position (cubic)         R2 = 0.027  #
  sentence category        R2 = 0.014  
  headroom  H(A_{i-1})     R2 = 0.351  #####################
  position + category      R2 = 0.039  ##
  headroom + position      R2 = 0.369  ######################
  all three                R2 = 0.373  ######################

  unique to headroom, over position+category : +0.334
  unique to category, over headroom+position : +0.004


**Expected:** headroom `R² ≈ 0.351`, position `0.027`, category `0.014`;
headroom uniquely adds `+0.334`, category `+0.004`.

### Is headroom just position in disguise?

If it were, the two would be near-collinear and the decomposition above would be
meaningless. They are not.

In [3]:
rho_kl_ent  = stats.spearmanr(d.entropy_before, d.kl_resampling).statistic
rho_ent_pos = stats.spearmanr(d.position, d.entropy_before).statistic
print(f"importance ~ headroom  : rho = {rho_kl_ent:+.3f}   <- strong")
print(f"headroom   ~ position  : rho = {rho_ent_pos:+.3f}   <- only moderate")
print("\nHeadroom does decline along the trace, but explains importance far")
print("better than position does. It is not position wearing a different hat.")

importance ~ headroom  : rho = +0.674   <- strong
headroom   ~ position  : rho = -0.448   <- only moderate

Headroom does decline along the trace, but explains importance far
better than position does. It is not position wearing a different hat.


## 1.2 The position-only control

A predictor that sees **only** normalised position and never reads the text. Fit
**leave-one-trace-out**, so it can never memorise the trace it scores.

The metric that matters in practice is not R² but **top-k recovery**: nobody uses
the importance scalar, they use it to pick the few sentences to look at.

In [4]:
from anchors.baselines import position_only_baseline

for measure in ["kl_resampling", "kl_counterfactual", "tv", "abs_delta_acc"]:
    sub = df.dropna(subset=[measure])
    res = position_only_baseline(sub.trace_id.values,
                                 sub.position.values,
                                 sub[measure].values.astype(float))
    t3, ch = res.topk_agreement.get(3, np.nan), res.topk_chance.get(3, np.nan)
    flag = "  <- at/below chance" if t3 <= ch else ""
    print(f"{measure:20s} rho={res.spearman:+.3f} (p={res.spearman_p:.2g})  "
          f"R2_oos={res.r2_oos:+.3f}  top3={t3:.3f} vs chance {ch:.3f}{flag}")

kl_resampling        rho=+0.044 (p=0.48)  R2_oos=-0.123  top3=0.083 vs chance 0.095  <- at/below chance
kl_counterfactual    rho=-0.058 (p=0.41)  R2_oos=-0.096  top3=0.042 vs chance 0.121  <- at/below chance
tv                   rho=+0.153 (p=0.014)  R2_oos=-0.092  top3=0.167 vs chance 0.095
abs_delta_acc        rho=+0.237 (p=0.00014)  R2_oos=-0.065  top3=0.250 vs chance 0.095


**Expected:** on `kl_resampling` — the paper's own metric — a text-blind
predictor lands at **0.083 against a chance rate of 0.095**, i.e. *below* chance.
Negative `R²_oos` means it is worse than predicting the mean.

**The honest complication** is the last row. `abs_delta_acc` is a difference of
two proportions and much less noisy than a KL between empirical distributions
over 2–3 answers — and it *does* show a positional effect (ρ ≈ +0.24).

Framed as a comparison of hypotheses: is that pattern more likely if (a) there is
no positional component, or (b) there is a *small* one that survives in the
low-variance statistic and drowns in the high-variance one? **(b).** So this
bounds a positional component rather than excluding it.

## 1.3 The noise floor — why you should distrust per-sentence numbers

With `R` rollouts a side, the empirical KL is positive **even when the two
distributions are identical**. That floor is estimated per sentence by parametric
bootstrap under the null that the sentence did nothing.

In [5]:
nf = summary["noise_floor"]
print(f"mean KL            {nf['mean_kl']:.4f}")
print(f"mean floor         {nf['mean_kl_null']:.4f}")
print(f"above own floor    {nf['frac_kl_above_null']:.1%}")
print("\nSo ~39% of sentences are indistinguishable from noise individually.")
print("The pooled and variance-decomposition results are the defensible ones;")
print("per-sentence rankings are not.")

below = (df.kl_resampling <= df.kl_null).sum()
print(f"\nsentences at or below their own floor: {below} / {len(df)}")

mean KL            0.0724
mean floor         0.0245
above own floor    61.0%

So ~39% of sentences are indistinguishable from noise individually.
The pooled and variance-decomposition results are the defensible ones;
per-sentence rankings are not.

sentences at or below their own floor: 99 / 254


## 1.4 The paraphrase control

The paper's **counterfactual importance** differs from plain resampling
importance by one thing: it keeps only rollouts whose replacement sentence was
semantically *different* from the original. The discarded half is a free,
on-policy paraphrase control.

If importance tracked content, replacements that mean the same thing should
diverge far less.

In [6]:
pc = summary["paraphrase_contrast"]
print(f"semantically DIFFERENT replacements : KL {pc['mean_kl_different']:.3f}")
print(f"semantically SIMILAR   replacements : KL {pc['mean_kl_similar']:.3f}")
print(f"gap (different - similar)           : {pc['mean_gap']:+.3f}")
print(f"95% CI (cluster bootstrap)          : "
      f"[{pc['gap_ci'][0]:+.3f}, {pc['gap_ci'][1]:+.3f}]")
print("\nThe filter that defines counterfactual importance has no detectable")
print("effect here; if anything the sign is reversed.")
print("\nCaveat that weakens this: each arm uses ~half the rollouts, so both are")
print("noisier than the pooled measure. This null is weaker than the position one.")

semantically DIFFERENT replacements : KL 0.103
semantically SIMILAR   replacements : KL 0.119
gap (different - similar)           : -0.015
95% CI (cluster bootstrap)          : [-0.081, +0.040]

The filter that defines counterfactual importance has no detectable
effect here; if anything the sign is reversed.

Caveat that weakens this: each arm uses ~half the rollouts, so both are
noisier than the pooled measure. This null is weaker than the position one.


---

## What this notebook does **not** show

- The **filler arm** was withdrawn: a pre-registered check found the filler sits
  −7.61 σ from the model's own sentences (a deliberately alien sentence is at
  −11.68 σ), so it measured *disruption*, not "same position, no content".
  See `data/filler_indistribution_*.json` and notebook 2.
- The **white-box** result is in notebook 4 — and it needed a correction.
- The **instrument validation** is notebook 3. Arguably you should read that one
  first: it is what makes the null above interpretable.